# Week 10a.1: Evaluating Agents

So far we have judged our agents by reading what they did: a report, a trace, a table of tool calls. That works for a demo. It does not tell you whether a change to the prompt made the agent better or worse, or whether it would handle the next hundred customers the way it handled the three you tried. For that, you need an evaluation: a fixed set of tasks, a way to decide whether each one succeeded, and a record of every run.

Today we'll evaluate a support agent at three levels:

| Level | Question | How it is checked | Example |
|---|---|---|---|
| Outcome | Did the agent leave the world in the right state? | code that compares the final state with the expected one | τ-bench compares the database after each task; SWE-bench runs the repository's tests |
| Trajectory | Did it get there the right way? | code that checks the sequence of tool calls | required steps, forbidden steps, order of steps |
| Quality | Is the reply good? | a rubric, applied by a person or by a model acting as a judge | LLM-as-a-judge |

Code checks are cheap and exact, and they cannot be argued with, so use them wherever the question allows. A judge is for what code cannot check, and a judge has to be checked itself before you trust it.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="claude-haiku-5-5")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

NOTE:

This notebook makes about 100 model calls: 16 agent runs of a few calls each, and about 50 calls to the judge. On the Gemini free tier, at 15 calls a minute, that is about seven minutes. If you see a 429 error, wait a minute and run the cell again.

## 1. Husky Tech Environment

To check outcomes, we need tasks with exactly one correct outcome. `husky_env.py` is a version of the Husky Tech store built for that.

- **Today's date is fixed** at November 13, 2026, so "delivered 30 days ago" means the same thing every time the notebook runs. With `date.today()`, as in our earlier support tools, a task that passed today could fail next week.
- **The agent's actions are recorded.** `issue_refund` and `escalate_to_human` write to `husky.env`. Nothing is refunded for real, but after each task we can read exactly what the agent did.
- **`husky.reset()` empties the record** before each task, so tasks do not affect each other.

In [ ]:
import json
import husky_env as husky

print("today:", husky.TODAY)
for order_id, order in husky.ORDERS.items():
    age = f"{husky.days_since_delivery(order_id):3} days ago" if order["status"] == "delivered" else order["status"]
    print(f"{order_id}  {order['item']:22} ${order['price']:8.2f}   {age}")

The agent's policy is in `husky.POLICY_PROMPT`. Every task's expected outcome follows from it.

In [ ]:
print(husky.POLICY_PROMPT)

### The tasks

`eval_tasks.json` has 16 tasks. Each one is a customer message and the end state it should produce: which orders are refunded, and whether the conversation is escalated.

In [ ]:
tasks = husky.tasks()

for t in tasks:
    print(f"{t['id']}  {t['category']:18} refunds={t['refunds'] or '-'}  escalate={t['escalate']}")
    print(f"     {t['message']}")

A few things about how the tasks were chosen.

- **They are spread over categories**, and most categories have more than one task. A single task per behavior tells you little, because one run of one task can pass by chance.
- **Some are at the boundaries of the policy.** HT-2004 was delivered exactly 30 days ago and HT-2005 31 days ago. Boundaries are where an agent's arithmetic, or its reading of "within 30 days", goes wrong.
- **Some test the rules about evidence.** In t09 and t16, the customer states a fact that the records contradict. In t14, the message tries to change the policy.
- **Some combine two requests**, where the agent has to apply a different rule to each.
- **Each one has a single correct outcome under the policy.** When we wrote them, any task we could argue either way was rewritten until it could not be. A task with a disputable answer measures the disagreement, not the agent.

Most of these tasks should be easy for a capable model. That is intended. A task set that the agent passes completely, or fails completely, cannot show a difference between two versions of the agent, and you want to find that out before you rely on it.

## 2. Running one task

The agent is the support loop from Week 4, with the environment's tools and the policy prompt.

In [ ]:
import time

from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition


def build_support(model, prompt=husky.POLICY_PROMPT):
    model_with_tools = model.bind_tools(husky.TOOLS)
    sys_msg = SystemMessage(content=prompt)

    def assistant(state: MessagesState) -> dict:
        return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])]}

    builder = StateGraph(MessagesState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(husky.TOOLS))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")
    return builder.compile()

In [ ]:
agent = build_support(model)

`run_task` resets the environment, runs one task, and returns a record of everything we will need to evaluate it: the final state, the reply, every tool call with the step it was made in, the tool results, the tokens used, and the time taken.

The record is plain data, so it can be saved to a file. In the next notebook, every run goes to a file, and the analysis reads the file.

In [ ]:
def evidence_of(messages):
    """Every tool call in a run and what it returned, one per line."""
    results = {m.tool_call_id: m.text for m in messages if m.type == "tool"}
    lines = []
    for m in messages:
        if m.type == "ai":
            for call in m.tool_calls:
                lines.append(f"{call['name']}({call['args']}) -> {results.get(call['id'], '(no result)')}")
    return "\n".join(lines) or "(no tool calls)"


def run_task(agent, task, config=None):
    """Run one task from a fresh environment and return a record of what happened."""
    husky.reset()
    start = time.time()
    out = agent.invoke({"messages": [HumanMessage(content=task["message"])]}, config or {})
    messages = out["messages"]
    calls, step = [], 0
    for m in messages:
        if m.type == "ai":
            step += 1
            calls += [{"step": step, "name": c["name"], "args": c["args"]} for c in m.tool_calls]
    return {
        "task": task["id"],
        "refunds": sorted(husky.env["refunds"]),
        "escalated": bool(husky.env["escalations"]),
        "reply": messages[-1].text,
        "calls": calls,
        "evidence": evidence_of(messages),
        "tokens": sum(m.usage_metadata["total_tokens"] for m in messages if getattr(m, "usage_metadata", None)),
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
record = run_task(agent, tasks[0])

print(json.dumps({k: v for k, v in record.items() if k != "evidence"}, indent=1))
print("\nevidence:\n" + record["evidence"])

### Tracing

If `LANGSMITH_TRACING=true` is in your `.env`, every run is also traced to LangSmith. A trace shows every model call with its full prompt, which the record does not. If you pass a name, tags, and metadata in the config, you can find a task's trace by its id later:

```python
run_task(agent, task, config={"run_name": f"task {task['id']}", "tags": ["w10_evals"], "metadata": {"task": task["id"]}})
```

When a task fails, read the trace before you change anything. It is the fastest way to see whether the failure came from the prompt, a tool, or the task itself.

## 3. Outcome

`check_outcome` compares the final state with the task's expected state and says what differs.

In [ ]:
def check_outcome(task, record):
    """Compare the environment's final state with the task's expected state."""
    problems = []
    if record["refunds"] != sorted(task["refunds"]):
        problems.append(f"refunded {record['refunds'] or 'nothing'}, expected {sorted(task['refunds']) or 'nothing'}")
    if record["escalated"] != task["escalate"]:
        problems.append("escalated, expected not to" if record["escalated"] else "did not escalate, expected to")
    return not problems, "; ".join(problems)

In [ ]:
ok, problem = check_outcome(tasks[0], record)

print(ok, ": ", problem)

Now all 16 tasks, once each.

In [ ]:
records = {}
for task in tasks:
    records[task["id"]] = run_task(agent, task, config={"run_name": f"task {task['id']}", "tags": ["w10a_evals"],
                                                        "metadata": {"task": task["id"]}})

passed = 0
for task in tasks:
    ok, problem = check_outcome(task, records[task["id"]])
    passed += ok
    print(f"{task['id']}  {task['category']:18} {'pass' if ok else 'FAIL  ' + problem}")
print(f"\noutcome: {passed}/{len(tasks)} tasks")

Read the failures, if there are any, and decide which kind each one is.

- **The agent broke the policy**, for example by trusting the customer's price in t16, or by following the "system notice" in t14. That is what the evaluation is for.
- **The task is wrong.** Its expected state does not follow from the policy. Fix the task, and note that you did.
- **The checker is wrong.** For example, it compares order ids without normalizing their case. Fix the checker.

Only the first kind is a result about the agent. Before you report a number, look at every failure at least once.

## 4. Trajectory

The policy says to check eligibility with `check_return_eligibility` before refunding. An agent that refunds the right order without checking passes the outcome check, but it skipped the step that would have caught an ineligible order.

`check_trajectory` checks that rule over the tool calls. It also catches a subtler case. When a model makes several tool calls in one message, `ToolNode` runs them together. A refund made in the same step as its eligibility check was decided before the check's result came back, so the check could not have informed it.

In [ ]:
def check_trajectory(record):
    """Every refund must come after an eligibility check of the same order in an earlier step."""
    checked, problems = {}, []
    for call in record["calls"]:
        order = str(call["args"].get("order_id", "")).strip().upper()
        if call["name"] == "check_return_eligibility":
            checked.setdefault(order, call["step"])
        if call["name"] == "issue_refund":
            if order not in checked:
                problems.append(f"refunded {order} without checking eligibility")
            elif checked[order] == call["step"]:
                problems.append(f"refunded {order} in the same step as its check, before seeing the result")
    return not problems, "; ".join(problems)

In [ ]:
for task in tasks:
    outcome_ok, _ = check_outcome(task, records[task["id"]])
    path_ok, problem = check_trajectory(records[task["id"]])
    steps = len(records[task["id"]]["calls"])
    print(f"{task['id']}  outcome {'pass' if outcome_ok else 'FAIL'}   trajectory {'pass' if path_ok else 'FAIL'}"
          f"   {steps} tool calls   {problem}")

Trajectory checks encode rules about the process, so they are only as good as the rules. Two cautions:

- **Do not require one exact sequence.** There are usually several acceptable orders of tool calls. Check what the policy actually requires, such as "check before refunding", and leave the rest free.
- **Report them separately from the outcome.** "Correct outcome, broke a rule on the way" and "wrong outcome" are different findings, and a single pass/fail number hides the difference.

The number of tool calls is also part of the trajectory. Two agents with the same outcomes, one of which makes twice as many calls, do not cost the same.

## 5. A judge for the reply

The outcome and the trajectory say nothing about the reply the customer reads. A reply can follow a correct refund and still state the wrong number of days, or promise money by tomorrow when nothing in the records supports that. Code cannot easily check this, so we use a rubric and a model to apply it.

The rubric has three criteria, each a yes-or-no question:

- **reason:** does the reply state the policy rule that decided each request?
- **grounded:** does every order fact in the reply match the evidence, with no unsupported promises?
- **action_matches:** does what the reply says was done match what the agent did?

Three things about how the judge is set up.

- **Yes-or-no criteria, not a 1 to 10 score.** Judges are inconsistent about what a 6 means, and two judges, or two runs of one judge, disagree more on a scale than on a yes-or-no question.
- **The explanation comes first in `Verdict`.** The model writes its fields in order, so it reasons about the reply before it commits to the verdicts.
- **The judge sees the evidence**, the tool calls and their results. Without it, the judge has no way to tell whether "delivered 10 days ago" is true.

In [ ]:
from pydantic import BaseModel, Field


class Verdict(BaseModel):
    explanation: str = Field(description="Two or three sentences on how the reply meets or misses each criterion.")
    reason: bool = Field(description="The reply states the policy rule that decided each request in the message.")
    grounded: bool = Field(description="Every order fact in the reply matches the evidence, and it promises nothing the evidence does not support.")
    action_matches: bool = Field(description="What the reply says was done matches the tool calls in the evidence.")


JUDGE_PROMPT = """You grade replies from a customer-support agent against a rubric.

You see the customer's message, the evidence (every tool the agent called and what it returned), and the
agent's reply. Judge only from what you are shown.

- reason: the reply states the policy rule that decided each request in the message, for example that an
  order is past the 30-day window, or that a refund over $300 needs a person.
- grounded: every order fact in the reply (dates, number of days, prices, status) matches the evidence, and
  the reply makes no promise the evidence does not support, such as when money will arrive.
- action_matches: what the reply says was done (refunded, escalated, declined) matches the tool calls in
  the evidence.

Write the explanation first, then the three verdicts."""

In [ ]:
from eval_harness import JUDGE_PROMPT_NO_EVIDENCE


def judge_reply(judge, message, reply, evidence=None):
    """Grade one reply. `judge` is a chat model; pass evidence=None to judge without the tool results."""
    grader = judge.with_structured_output(Verdict, method="json_schema")
    if evidence is None:
        prompt = JUDGE_PROMPT_NO_EVIDENCE
        content = f"Customer message:\n{message}\n\nAgent reply:\n{reply}"
    else:
        prompt = JUDGE_PROMPT
        content = f"Customer message:\n{message}\n\nEvidence:\n{evidence}\n\nAgent reply:\n{reply}"
    return grader.invoke([SystemMessage(content=prompt), HumanMessage(content=content)])

`judge_reply` also has a mode without the evidence, which we'll use in the next section. Its prompt, `JUDGE_PROMPT_NO_EVIDENCE`, is the same rubric for a judge that sees only the message and the reply.

We use the same model as the agent here, so the notebook runs with one model. If you can, use a different and stronger model as the judge. Judges tend to rate text from their own model family more favorably.

In [ ]:
from langchain.chat_models import init_chat_model

judge = init_chat_model(model="claude-haiku-5-5")

# judge = model

In [ ]:
verdict = judge_reply(judge, tasks[0]["message"], records["t01"]["reply"], records["t01"]["evidence"])
print(records["t01"]["reply"], "\n")
print(verdict)

## 6. Checking the judge

Before we use the judge's verdicts as a measurement, we need to know how often they are right. The way to find out is to compare the judge with human labels on replies where the right answer is known.

`judge_calibration.json` has 18 replies to tasks from our set, each with the evidence from a real run of the tools. We wrote the replies for this notebook. Six are correct, and each of the others has one planted defect, such as a wrong number of days, a promise the records do not support, or a claim to have refunded when the agent escalated. The labels record which criteria each reply meets.

In [ ]:
calibration = json.load(open("judge_calibration.json"))

for c in calibration[:4]:
    print(f"{c['id']}  {c['labels']}  defect: {c['defect']}")
    print(f"     {c['reply']}\n")

### Agreement
Cohen's kappa is used to quantify interrater agreement

We judge all 18, with the evidence and without it, and compare each criterion with the labels.

Raw agreement can look good by accident. If 16 of 18 replies meet a criterion, a judge that always says yes agrees 89% of the time. **Cohen's kappa** corrects for this. It compares the observed agreement with the agreement expected by chance, given how often each side says yes:

kappa = (observed − chance) / (1 − chance)

A kappa of 1 is perfect agreement, 0 is no better than chance, and a negative kappa is worse than chance. As a rough guide, above 0.8 is strong and below 0.6 is not enough to rely on.

In [ ]:
def cohen_kappa(a, b):
    """Agreement between two lists of labels, corrected for the agreement expected by chance."""
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b)) / n
    labels = set(a) | set(b)                                            # e.g. {"PASS", "FAIL"}
    chances = [(a.count(label) / n) * (b.count(label) / n) for label in labels]     # one per label: both pick it by chance
    chance = sum(chances)                                               # e.g. 0.49 + 0.09 = 0.58
    if chance == 1:                             # one label everywhere: kappa is undefined
        return float("nan")
    
    kappa = (observed - chance) / (1 - chance)
    return kappa

Here is the function on a small example.

- A person and a judge each labeled ten replies as `PASS` or `FAIL` on one criterion. 
- They agree on 8 of the 10. 
- The person said `PASS` on 7 replies and so did the judge, so two raters labeling at those rates by chance would agree on 0.7 × 0.7 + 0.3 × 0.3 = 0.58 of the replies. 
- Kappa is (0.80 − 0.58) / (1 − 0.58) = 0.52. 
- The last line checks a judge that always says `PASS`: it agrees on 7 of the 10, and its kappa is 0. 
- The function works with any labels, so it also takes the `True`/`False` verdicts of the judge below.

In [ ]:
person = ["PASS", "FAIL", "PASS", "PASS", "PASS", "FAIL", "PASS", "PASS", "FAIL", "PASS"]
judged = ["PASS", "FAIL", "PASS", "FAIL", "PASS", "PASS", "PASS", "PASS", "FAIL", "PASS"]

print(f"kappa: {cohen_kappa(person, judged):.2f}")

### Calibrating the judge

In [ ]:
c = calibration[0]
c

In [ ]:
judge_reply(judge, c["message"], c["reply"], c["evidence"])

In [ ]:
CRITERIA = ["reason", "grounded", "action_matches"]

verdicts = {"with evidence": [], "without evidence": []}

for c in calibration:
    verdicts["with evidence"].append(judge_reply(judge, c["message"], c["reply"], c["evidence"]))
    verdicts["without evidence"].append(judge_reply(judge, c["message"], c["reply"], None))


In [ ]:
verdicts

In [ ]:
verdicts.items()

In [ ]:
human = {k: [c["labels"][k] for c in calibration] for k in CRITERIA}
print(f"{'':18}{'criterion':16}{'agreement':>10}{'kappa':>8}")

for mode, vs in verdicts.items():
    for k in CRITERIA:
        judged = [getattr(v, k) for v in vs]
        agree = sum(h == j for h, j in zip(human[k], judged))
        print(f"{mode:18}{k:16}{agree:>6}/{len(judged)}{cohen_kappa(human[k], judged):>8.2f}")

Without the evidence, the judge can still tell whether a reply gives a reason, but it cannot check facts or actions. It tends to accept whatever the reply claims, so kappa for `grounded` and `action_matches` falls toward zero. A judge that does not see the evidence is checking whether the reply sounds right.

Now the disagreements with the evidence, which are the ones that matter:

In [ ]:
for c, v in zip(calibration, verdicts["with evidence"]):
    missed = [k for k in CRITERIA if getattr(v, k) != c["labels"][k]]
    if missed:
        print(f"{c['id']}  missed {missed}   defect: {c['defect']}")
        print(f"     reply: {c['reply']}")
        print(f"     judge: {v.explanation}\n")

Read the judge's explanation for each disagreement. Some will be the judge's mistakes. Others show that the rubric is ambiguous. For example, when a reply claims a refund that never happened, is that a failure of `grounded`, of `action_matches`, or of both? The rubric does not say, so the judge and the labels can reasonably differ. When that happens, fix the rubric and label again. Do not tune the judge prompt until it matches your labels on these 18 replies, or you will have a judge that agrees with you on the calibration set and nowhere else.

For your project, label replies from your own agent, not ones written for the purpose, and have two people label them independently. Their agreement with each other, as kappa, is the ceiling for the judge: a judge cannot be held to a standard that two people do not meet. On replies where the two people disagree, the rubric needs work.

## 7. Judging the agent's replies

With the judge checked, we can use it on the agent's 16 replies.

In [ ]:
judged = {}
for task in tasks:
    r = records[task["id"]]
    judged[task["id"]] = judge_reply(judge, task["message"], r["reply"], r["evidence"])

for k in CRITERIA:
    print(f"{k:16} {sum(getattr(v, k) for v in judged.values())}/{len(judged)} replies")

for task_id, v in judged.items():
    missed = [k for k in CRITERIA if not getattr(v, k)]
    if missed:
        print(f"\n{task_id}  fails {missed}\n     {records[task_id]['reply']}\n     judge: {v.explanation}")

Treat these numbers with the uncertainty the calibration showed. If the judge misses some defects on a criterion, a 15/16 on that criterion may hide a reply or two that a person would fail. Report the judge's agreement with human labels next to any number it produces.

For an example, compare the calibration with the replies. If the judge passed a reply in the calibration set that gives no reason, such as c03, look at the agent's reply to t01 in section 2. If it also gives no reason and the judge passed it on `reason`, the leniency the calibration showed is in these numbers too.

## 8. All three levels

Here is the whole evaluation of this run in one table.

In [ ]:
print(f"{'task':6}{'category':20}{'outcome':9}{'trajectory':12}{'reply':7}")
for task in tasks:
    r = records[task["id"]]
    outcome_ok, _ = check_outcome(task, r)
    path_ok, _ = check_trajectory(r)
    reply_ok = all(getattr(judged[task["id"]], k) for k in CRITERIA)
    mark = lambda ok: "pass" if ok else "FAIL"
    print(f"{task['id']:6}{task['category']:20}{mark(outcome_ok):9}{mark(path_ok):12}{mark(reply_ok):7}")

| Level | What it catches | What it misses | Cost |
|---|---|---|---|
| Outcome | wrong refunds, missing escalations | a right outcome reached by breaking a rule; a bad reply | none beyond the run |
| Trajectory | skipped checks, decisions made before the evidence arrived | anything the rules do not mention | none beyond the run |
| Judge | wrong facts and false claims in the reply, missing reasons | whatever the judge gets wrong, which you measured in section 6 | one model call per reply |

One run of 16 tasks is a single sample. In actual evals, you'd run every task several times, under two versions of the agent, and ask whether the difference between them is larger than the run-to-run variation.

## 9. Things to try

1. Add a task of your own for a case the set does not cover, such as a customer who asks for a refund on two orders that are both over $300. Write the expected state first, then run it. Is there exactly one correct outcome under the policy?

2. Break the checker on purpose: change `check_outcome` so that it compares `record["refunds"]` with `task["refunds"]` without sorting. Which tasks now fail, and would you have noticed if you had only looked at the success rate?

3. Write a fourth criterion for the rubric, such as "the reply offers the warranty when it declines a refund for an order past the window". Write five replies that meet it and five that do not, and measure the judge's kappa on them.

4. Label the agent's 16 replies yourself on the three criteria before you look at the judge's verdicts in section 7. Then compute kappa between your labels and the judge's. Is it close to what the calibration set showed?

5. Use a different model as the judge, for example by running one of the other model cells into a variable named `judge`. Does kappa on the calibration set change?

In [ ]:
# your experiments here